!pip install pyneid hiresprv dace-query

In [ ]:
from exotic.api.rv_archive import query_harps, query_dace, query_espresso, query_neid, query_sophie

# query_dace("KELT-6")
# query_harps("WASP-77")
# query_espresso("WASP-77")
# query_neid("GJ 3470")
# query_sophie("GPX-1 b")

In [ ]:
from exotic.api.ew import ExoplanetWatch
from IPython.display import display, Image

EW = ExoplanetWatch()
EW.target_list

In [ ]:
import json

# names are case and space sensitive
target = 'GJ 436 b'

result = EW.get(target)

print( json.dumps(result.priors, indent=4) )

display(Image(url=result.observations[0].lightcurve_url))

result.observations[0].parameters

In [ ]:
from exotic.api.rv_archive import RadialVelocityDatabase

rv_db = RadialVelocityDatabase(target)

# Get the collected data
print(rv_db.data.head())

# Print summary statistics
rv_db.summary_statistics()

# Plot the data
rv_db.plot_rv_data()

# Save data to CSV
rv_db.save_to_csv(f"{target}_rv_data.csv")

In [ ]:
result.priors.keys()

In [ ]:
import matplotlib.pyplot as plt
from exotic.api.rv_fitter import rv_fitter

# logg[cm/s2], rs[rsun]
stellar_mass = lambda logg,rs: ((rs*6.955e10)**2 * 10**logg / 6.674e-8)/1000./1.989e30

# need RV prior dict in specific keys
prior = dict(**result.observations[0].parameters)
prior['rstar'] = float(result.priors['R*']['value'])
prior['logg'] = float(result.priors['LOGG']['value'])
prior['mstar'] = stellar_mass(prior['logg'], prior['rstar'])
prior['mplanet'] = 1 # M_Jupiter # TODO FILL IN MANUALLY
prior['rv_linear'] = 0 # m/s/day
prior['rv_quad'] = 0 # m/s/day^2


# bounds for optimization - shared between all datasets
global_bounds = {
    'per':[prior['per']-0.005,prior['per']+0.005],
    'omega':[prior['omega']-180,prior['omega']+180],
    'ecc':[0, prior['ecc']+0.2],
    'rv_linear':[-0.05,0.05], # m/s/day
    'mplanet':[0, prior['mplanet']*2],
}

# alloc data
rv_data = []
local_rv_bounds = []

# myfit = rv_fitter(rv_data, global_bounds, local_rv_bounds, verbose=False)

# loop over telescopes
for tele in rv_db.data.ins_name.unique():
    # get data for this telescope
    df_tel = rv_db.data[rv_db.data.ins_name == tele]

    if tele == 'SOPHIE':
        # some data cleaning may be needed
        # import numpy as np  # Needed only for this optional filter.
        #df_tel = df_tel[np.abs(df_tel['rv']) > 9625]

        # just skip it
        continue

    # create data dict
    one_data = {
        'time':df_tel['bjd'].values,
        'vel':df_tel['rv'].values,
        'velerr':df_tel['rv_err'].values,
        'priors':prior.copy(),
        'name':tele
    }

    # fit individual dataset to estimate errors
    #if len(one_data['time']) > 10:
    #    myfit = rv_fitter([one_data], global_bounds, [{}], verbose=False)
    #    one_data['velerr'] *= myfit.data[0]['error_scale']

    # add to list for fitter
    rv_data.append(one_data)

    # local bounds are applied to each dataset separately
    print(f"{tele} has {len(rv_data[-1]['time'])} points")
    local_rv_bounds.append({})

print(prior)

# bounds for optimization
global_bounds = {
    'per':[prior['per']-0.005,prior['per']+0.005],
    'omega':[prior['omega']-180,prior['omega']+180],
    'ecc':[0, prior['ecc']+0.2],
    'rv_linear':[-0.1,0.1], # m/s/day
    'mplanet':[0, prior['mplanet']*2],
}

myfit = rv_fitter(rv_data, global_bounds, local_rv_bounds, verbose=True)

# print Bayesian evidence
print(myfit.results['logz'], myfit.results['logzerr'])

# corner plot
myfit.plot_triangle()
plt.tight_layout()
plt.savefig('RV_triangle.png')
plt.show()

# best fit
myfit.plot_bestfit()
plt.tight_layout()
plt.savefig('RV_bestfit.png')
plt.show()